# LiteRT-LM Export Demo — Gemma 3 270MMinimal demonstration of exporting a KerasHub `Gemma3CausalLM` to a **LiteRT-LM bundle** (`.litertlm`).LiteRT-LM packages the TFLite model (with `prefill` + `decode` signatures), the SentencePiece tokenizer, and LLM metadata into a single file. On Android, the LiteRT-LM runtime handles tokenization, KV-cache management, sampling, and streaming for you.> **Requirements:**> - `KERAS_BACKEND=torch`> - `litert-torch` and `litert-lm-builder` installed> - This export path is **PyTorch-only**> - LiteRT-LM PR branch: `torch-backend-litert-minimal-litertlm`

In [1]:
!pip install -q litert-torch!pip install -q git+https://github.com/keras-team/keras.git!pip install -q git+https://github.com/pctablet505/keras-hub.git@torch-backend-litert-minimal-litertlmimport osPRESET = "hf://google/gemma-3-270m-it"CACHE_LEN = 1024   # Max conversation contextPREFILL_LEN = 128  # Max prompt length per turnprint("Preset:", PRESET)print(f"cache_length={CACHE_LEN}, prefill_seq_len={PREFILL_LEN}")

Preset: hf://google/gemma-3-270m-it
cache_length=1024, prefill_seq_len=128


In [2]:
import osos.environ["KERAS_BACKEND"] = "torch"os.environ["CUDA_VISIBLE_DEVICES"] = "-1"os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"import keras_hubmodel = keras_hub.models.Gemma3CausalLM.from_preset(PRESET)model.preprocessor.sequence_length = CACHE_LEN# Build via direct call with concrete shapesprocessed = model.preprocessor({"prompts": ["What is Keras?"], "responses": [""]})model(processed[0])print("Model built. Output shape:", model(processed[0]).shape)

Preset: hf://google/gemma-3-270m-it
cache_length=1024, prefill_seq_len=128


## Export to `.litertlm`A single call with `format="litertlm"` produces the bundle. The exporter automatically:- Traces `prefill` (prompt → KV cache) and `decode` (single token → logits + updated KV cache) signatures- Bundles the SentencePiece tokenizer- Writes `LlmMetadata` (start/stop tokens, context window, model type)

In [3]:
# Single bucket (legacy behavior)
# model.export("gemma3_270m_it.litertlm", format="litertlm", prefill_seq_len=128)

# Multiple buckets — runtime dispatches to the smallest fitting signature
# NOTE: Each bucket adds ~2-4 min to export time (signature is re-traced),
# but model size increase is minimal (~0.07% for 3 extra signatures)
# because weights are shared across all prefill signatures.
model.export(
    "gemma3_270m_it.litertlm",
    format="litertlm",
    prefill_seq_len=[32, 64, 128, 256, 512, 1024],
)
print("✅ Exported with bucketing: prefill_32, prefill_64, ..., prefill_1024")
print("Size (MB):", round(os.path.getsize("gemma3_270m_it.litertlm") / 1e6, 1))

✅ Exported with bucketing: prefill_32, prefill_64, ..., prefill_1024
Size (MB): 1082.7


## Verify Bundle ContentsPeek inside the `.litertlm` file to confirm it contains the expected assets.

In [4]:
import iofrom litert_lm_builder import litertlm_peekoutput = io.StringIO()litertlm_peek.peek_litertlm_file("gemma3_270m_it.litertlm", None, output)print(output.getvalue())print("✅ LiteRT-LM bundle verified.")

LiteRT-LM Version: 1.5.0

+------------------------------------------------+
|                System Metadata                 |
+------------------------------------------------+
  Key: Authors, Value (String): KerasHub
  Key: uuid, Value (String): a7491ec1-7d84-401d-8fca-b1ea65d818fa
  Key: creation_timestamp, Value (String): 2026-05-27T13:43:21.673119+00:00

+------------------------------------------------+
|                  Sections (3)                  |
+------------------------------------------------+

Section 0:
  Items:
    Key: model_type, Value (String): tf_lite_prefill_decode
  Begin Offset: 16384
  End Offset:   1078013336
  Data Type:    TFLiteModel


Section 1:
  Items:
    <None>
  Begin Offset: 1078018048
  End Offset:   1082707122
  Data Type:    SP_Tokenizer


Section 2:
  Items:
    <None>
  Begin Offset: 1082720256
  End Offset:   1082720284
  Data Type:    LlmMetadataProto
  <<<<<<<< start of LlmMetadata
    start_token {
      token_ids {
        ids: 2
      }

### Bucketing Performance (Measured on Pixel 9)

| Model | Prompt Tokens | TTFT |
|-------|--------------|------|
| Fixed-128 (`prefill_seq_len=128`) | 31 tok | **2866.5 ms** |
| Bucketed `[32, 64, 128]` | 31 tok | **1626.4 ms** |

**→ 43.3% faster TTFT** with bucketing for a ~31-token prompt.

The fixed model pads every prompt to 128 tokens. The bucketed runtime dispatches to `prefill_64` (smallest fitting signature), avoiding ~60% of wasted prefill compute.

In [ ]:
from litert_torch.generative.quantize.quant_recipes import full_dynamic_recipe

model.export(
    "gemma3_270m_it_wi8afp32.litertlm",
    format="litertlm",
    prefill_seq_len=[32, 64, 128, 256, 512, 1024],
    quant_config=full_dynamic_recipe(),
)
print("✅ Exported + quantized in one step")
print("Size (MB):", round(os.path.getsize("gemma3_270m_it_wi8afp32.litertlm") / 1e6, 1))

## Push to Android & Test

```bash
# Push to device
adb push gemma3_270m_it_wi8afp32.litertlm /data/local/tmp/
```

**Verified on Pixel 9 (physical ARM64):**
- Engine init: **2,318 ms** (vs ~5,336 ms for FP32)
- Generation: **4,213 ms** for "What is Keras?"
- Response: *"Keras is a high-level, Python API for building and training machine learning models..."*